This notebook tests 4 mamba architeture, puremamba, hybrid with resnet pretrain, with regulization, desnet pretrain. It then compares these model performance, genralizability. The a segementation head is added to these models for explainability. In addition Grad-Cam is used for explainability.

In [ ]:
# Install dependencies
!pip install segmentation-models-pytorch
!pip install torchmetrics==0.11.0
!pip install opencv-python
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install ninja
!pip install einops timm
!pip install thop -q
!pip install scikit-learn scikit-image pandas matplotlib seaborn tqdm
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

print("Installation complete! Please restart runtime now.")

In [ ]:
# Imports
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from einops import rearrange
from tqdm import tqdm
from google.colab import drive
import zipfile
import glob
import time
from functools import wraps
import json
from datetime import datetime
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_curve, auc, jaccard_score
from sklearn.model_selection import train_test_split
import cv2
import shutil
from PIL import Image
from scipy import ndimage
import random
import segmentation_models_pytorch as smp
from torchvision.models import resnet50, ResNet50_Weights
import timm
from IPython.display import display, Javascript
import threading
import torchmetrics
from scipy.ndimage import binary_dilation, label as ndimage_label
from thop import profile
from sklearn.metrics import roc_auc_score
from collections import defaultdict
import matplotlib.patches as patches
from scipy.ndimage import zoom

# Import mamba-ssm
try:
    from mamba_ssm import Mamba
    from mamba_ssm.models.config_mamba import MambaConfig
    print("mamba-ssm imported successfully")
except ImportError as e:
    print(f"Import error: {e}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Loading Data
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/MINI-DDSM-Complete-JPEG-8.zip'

# Check if the zip file exists
if os.path.exists(zip_path):
    print(f"✓ Found dataset at: {zip_path}")
    print(f"File size: {os.path.getsize(zip_path) / (1024**3):.2f} GB")

    # Create extraction directory
    extract_path = '/content/MINI-DDSM'
    os.makedirs(extract_path, exist_ok=True)

    # Extract the dataset
    print("Extracting dataset... (this may take 5-10 minutes)")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✓ Extraction complete!")

    # Verify extraction
    print("\nDataset extracted to:", extract_path)

In [ ]:
# Data Processing

mid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Helper Functions

In [ ]:
# At the top of your script, right after imports:
def set_reproducibility(seed=42):
    """Set all random seeds for reproducible results"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make CuDNN deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Additional for Python hash randomization
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Call it immediately
set_reproducibility(42)

In [ ]:

# Enhanced Early Stopping Class that monitors both accuracy and loss
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, monitor='loss'):

        self.patience = patience
        self.min_delta = min_delta
        self.monitor = monitor
        self.counter = 0
        self.best_loss = None
        self.best_acc = None
        self.early_stop = False

    def __call__(self, val_loss, val_acc):
        if self.monitor == 'loss':
            # Monitor validation loss (lower is better)
            if self.best_loss is None:
                self.best_loss = val_loss
            elif val_loss > self.best_loss - self.min_delta:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_loss = val_loss
                self.counter = 0

        elif self.monitor == 'accuracy':
            # Monitor validation accuracy (higher is better)
            if self.best_acc is None:
                self.best_acc = val_acc
            elif val_acc < self.best_acc + self.min_delta:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_acc = val_acc
                self.counter = 0

        elif self.monitor == 'both':
            # Monitor both: stop if no improvement in either metric
            improved = False

            # Check loss improvement
            if self.best_loss is None:
                self.best_loss = val_loss
                improved = True
            elif val_loss < self.best_loss - self.min_delta:
                self.best_loss = val_loss
                improved = True

            # Check accuracy improvement
            if self.best_acc is None:
                self.best_acc = val_acc
                improved = True
            elif val_acc > self.best_acc + self.min_delta:
                self.best_acc = val_acc
                improved = True

            if improved:
                self.counter = 0
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True

In [ ]:
#Helper Class
class ModelCheckpointer:
    def __init__(self, base_dir='/content/model_checkpoints'):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

    def save_checkpoint(self, model, model_name, epoch, optimizer=None,
                      metrics=None, is_best=False):
        checkpoint = {
            'model_name': model_name,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
            'metrics': metrics or {},
        }

        # Define path outside the if block - THIS IS THE KEY FIX
        path = os.path.join(self.base_dir, f"{model_name}_epoch_{epoch}.pth")
        torch.save(checkpoint, path)

        if is_best:
            best_path = os.path.join(self.base_dir, f"{model_name}_best.pth")
            torch.save(checkpoint, best_path)
            print(f"Saved best {model_name} model")

        return path  # Now path is always defined

    def load_checkpoint(self, model_name, best=True):
        if best:
            path = os.path.join(self.base_dir, f"{model_name}_best.pth")
        else:
            files = [f for f in os.listdir(self.base_dir) if f.startswith(model_name)]
            if not files:
                return None
            latest = sorted(files)[-1]
            path = os.path.join(self.base_dir, latest)

        if os.path.exists(path):
            return torch.load(path)
        return None

checkpointer = ModelCheckpointer()

In [ ]:
# Tor Prevent Crashing Helper Function
def keep_colab_active():
    def click_connect_button():
      js_code = """
        function ClickConnect(){
            console.log("Keeping Colab active...");
            document.querySelector("colab-connect-button").click()
        }
        setInterval(ClickConnect, 60000)
        """
      display(Javascript(js_code))

    # Also keep Python busy
    def keep_alive():
        while True:
            time.sleep(30)

    threading.Thread(target=keep_alive, daemon=True).start()
    click_connect_button()
    print("Keep-alive mechanism activated - Colab will stay connected")

In [ ]:
# Time Comparison

def timer(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"⏱️ {func.__name__} took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
        return result, elapsed
    return wrapper

Model Definition

In [ ]:
class PureMamba(nn.Module):
    def __init__(self, img_size=256, patch_size=16, num_classes=2):

      super().__init__()
      self.num_patches = (img_size // patch_size) ** 2
      self.patch_embed = nn.Conv2d(3, 384, kernel_size=patch_size, stride=patch_size)

      self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, 384) * 0.02)
      self.pos_drop = nn.Dropout(0.1)

      self.mamba_layers = nn.ModuleList([
          Mamba(d_model=384, d_state=16, d_conv=4, expand=2,
                dt_rank='auto', bias=False, conv_bias=True) for _ in range(8)
      ])

      self.norms = nn.ModuleList([nn.LayerNorm(384) for _ in range(8)])
      self.final_norm = nn.LayerNorm(384)

      # Conv head
      self.conv_head = nn.Sequential(
          nn.Conv1d(384, 256, kernel_size=3, padding=1),
          nn.BatchNorm1d(256),
          nn.ReLU(inplace=True),
          nn.Dropout(0.3),
          nn.AdaptiveAvgPool1d(1),  # Global pooling
          nn.Flatten(),
          nn.Linear(256, num_classes)
      )

    def forward(self, x):
      x = self.patch_embed(x)
      x = x.flatten(2).transpose(1, 2)  # [B, 256, 384]
      x = x + self.pos_embed
      x = self.pos_drop(x)

      for mamba, norm in zip(self.mamba_layers, self.norms):
          x = x + mamba(norm(x))        # can inline x_norm too

      x = self.final_norm(x)
      x = x.transpose(1, 2)
      return self.conv_head(x)

In [ ]:
class HybridMamba(nn.Module):
    def __init__(self, num_classes=2, num_seg_classes=2):
        super().__init__()

        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.classification_model = nn.Module()
        self.classification_model.encoder = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )

        self.classification_model.projection = nn.Conv2d(1024, 384, kernel_size=1)
        self.classification_model.mamba = Mamba(d_model=384, d_state=16, d_conv=4, expand=2,
                                               dt_rank='auto', bias=False, conv_bias=True)
        self.classification_model.pos_embed = nn.Parameter(torch.randn(1, 256, 384) * 0.02)
        self.classification_model.norm = nn.LayerNorm(384)

        # Conv head
        self.conv_head = nn.Sequential(
            nn.Conv1d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.AdaptiveAvgPool1d(1),  # Global pooling
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.classification_model.encoder(x)
        features = self.classification_model.projection(features)
        B, D, H, W = features.shape
        seq = features.flatten(2).transpose(1, 2) + self.classification_model.pos_embed
        seq = self.classification_model.mamba(seq)
        seq = self.classification_model.norm(seq)

        # Reshape back to 2D for conv head
        seq_2d = seq.transpose(1, 2)
        return self.conv_head(seq_2d)

In [ ]:
class HybridMambaRegularized(nn.Module):
    def __init__(self, num_classes=2, num_seg_classes=2):
        super().__init__()

        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.classification_model = nn.Module()
        self.classification_model.encoder = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )

        # Freeze early layers
        for name, param in self.classification_model.encoder.named_parameters():
            if 'layer3' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.classification_model.projection = nn.Conv2d(1024, 384, kernel_size=1)
        self.classification_model.projection_dropout = nn.Dropout2d(0.2)
        self.classification_model.mamba = Mamba(d_model=384, d_state=8, d_conv=4, expand=2,
                                               dt_rank='auto', bias=False, conv_bias=True)
        self.classification_model.pos_embed = nn.Parameter(torch.randn(1, 256, 384) * 0.02)
        self.classification_model.mamba_dropout = nn.Dropout(0.3)

        self.classification_model.norm = nn.LayerNorm(384)

        #  Conv head
        self.conv_head = nn.Sequential(
            nn.Conv1d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.AdaptiveAvgPool1d(1),  # Global pooling
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.classification_model.encoder(x)
        features = self.classification_model.projection(features)
        features = self.classification_model.projection_dropout(features)
        B, D, H, W = features.shape
        seq = features.flatten(2).transpose(1, 2) + self.classification_model.pos_embed
        seq = self.classification_model.mamba(seq)
        seq = self.classification_model.norm(seq)
        seq = self.classification_model.mamba_dropout(seq)
        seq = seq.transpose(1, 2)
        return self.conv_head(seq)

In [ ]:
class DenseNetMamba(nn.Module):
    def __init__(self, num_classes=2, num_seg_classes=2):
        super().__init__()

        self.classification_model = nn.Module()
        self.classification_model.encoder = timm.create_model("densenet121", pretrained=True, features_only=True)

        for name, param in self.classification_model.encoder.named_parameters():
            if "denseblock2" in name or "denseblock3" in name or "denseblock4" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.classification_model.projection = nn.Conv2d(1024, 384, kernel_size=1)
        self.classification_model.projection_dropout = nn.Dropout2d(0.2)
        self.classification_model.mamba = Mamba(d_model=384, d_state=16, d_conv=4, expand=2,
                                               dt_rank='auto', bias=False, conv_bias=True)
        self.classification_model.mamba_dropout = nn.Dropout(0.3)
        self.classification_model.pos_embed = nn.Parameter(torch.randn(1, 64, 384) * 0.02)
        self.classification_model.norm = nn.LayerNorm(384)

        # Conv head with residual connection
        self.conv_head = nn.Sequential(
            nn.Conv1d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.AdaptiveAvgPool1d(1),  # Global pooling
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.classification_model.encoder(x)[-1]
        features = self.classification_model.projection(features)
        features = self.classification_model.projection_dropout(features)
        B, C, H, W = features.shape
        seq = features.flatten(2).transpose(1, 2) + self.classification_model.pos_embed
        seq = self.classification_model.mamba(seq)
        seq = self.classification_model.norm(seq)
        seq = self.classification_model.mamba_dropout(seq)
        return self.conv_head(seq.transpose(1, 2))

In [ ]:
# Model configurations
models_config = [
      {
        'name': 'PureMamba',
        'class': PureMamba,
        'params': {'img_size': 256, 'patch_size': 16, 'num_classes': 2},
        'transform': 'mid'
    },

    {
        'name': 'HybridMamba',
        'class': HybridMamba,
        'params': {'num_classes': 2},
        'transform': 'mid'
    },
    {
        'name': 'HybridMambaRegularized',
        'class': HybridMambaRegularized,
        'params': {'num_classes': 2},
        'transform': 'mid'
    },
    {
        'name': 'DenseNetMamba',
        'class': DenseNetMamba,
        'params': {'num_classes': 2},
        'transform': 'mid'
    }
]

In [ ]:
# Quick FLOPs check (add after models_config)
for config in models_config:
    model = config['class'](**config['params']).to('cuda')
    dummy = torch.randn(1, 3, 256, 256).to('cuda')
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    print(f"{config['name']}: {params/1e6:.1f}M params, {flops/1e9:.2f}G FLOPs")

Classification Training

In [ ]:
# Training Function with Early Stopping monitoring both metrics
@timer
def train_model(model, model_name, train_loader, val_loader, epochs=15, device='cuda'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    # Early stopping monitoring BOTH accuracy and loss with patience=5
    early_stopping = EarlyStopping(patience=5, min_delta=0.001, monitor='both')

    best_acc = 0.0
    best_loss = float('inf')
    train_loss_history, train_acc_history = [], []
    val_loss_history, val_acc_history = [], []

    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            train_bar.set_postfix({'loss': f'{train_loss/(train_bar.n+1):.4f}',
                                  'acc': f'{100.*correct/total:.2f}%'})

        train_acc = 100. * correct / total
        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_correct, val_total, val_loss = 0, 0, 0.0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

                val_bar.set_postfix({'acc': f'{100.*val_correct/val_total:.2f}%'})

        val_acc = 100. * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)

        scheduler.step(val_acc)

        train_loss_history.append(avg_train_loss)
        train_acc_history.append(train_acc)
        val_loss_history.append(avg_val_loss)
        val_acc_history.append(val_acc)

        print(f'\nEpoch {epoch+1} - Train Acc: {train_acc:.2f}%, Train Loss: {avg_train_loss:.4f}')
        print(f'            Val Acc: {val_acc:.2f}%, Val Loss: {avg_val_loss:.4f}')

        # Check for best model based on validation accuracy
        is_best = val_acc > best_acc
        if is_best:
            best_acc = val_acc
            best_loss = avg_val_loss
            print(f'New best accuracy! {best_acc:.2f}%')

        metrics = {
            'train_acc': train_acc, 'val_acc': val_acc,
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss
        }
        checkpointer.save_checkpoint(model, model_name, epoch, optimizer, metrics, is_best)

        # Early stopping monitoring both metrics
        early_stopping(avg_val_loss, val_acc)

        if early_stopping.early_stop:
            print(f"\n Early stopping triggered at epoch {epoch+1}")
            print(f"   No improvement in validation accuracy or loss for {early_stopping.patience} consecutive epochs")
            print(f"   Best accuracy: {best_acc:.2f}% | Best loss: {best_loss:.4f}")
            break

    history = {
        'train_acc': train_acc_history,
        'val_acc': val_acc_history,
        'train_loss': train_loss_history,
        'val_loss': val_loss_history,
        'best_acc': best_acc,
        'best_loss': best_loss,
        'stopped_epoch': epoch if early_stopping.early_stop else None
    }

    with open(f'/content/model_checkpoints/{model_name}_history.json', 'w') as f:
        json.dump(history, f)

    print(f"\n{'='*50}")
    print(f"{model_name} complete!")
    print(f"Best accuracy: {best_acc:.2f}%")
    print(f"Best loss: {best_loss:.4f}")
    print(f"{'='*50}")

    return history

In [ ]:
def evaluate_model(model, model_name, test_loader, device='cuda'):
    print("\n" + "="*50)
    print(f"FINAL MODEL EVALUATION - {model_name}")
    print("="*50 + "\n")

    model.eval()

    # Collect predictions
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating on TEST set"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # Calculate metrics with safe handling
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)

    print("\n" + "="*50)
    print("CLASSIFICATION METRICS (TEST SET)")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")

    class_names = ['Benign', 'Malignant']
    print("\nCLASSIFICATION REPORT:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    tn, fp, fn, tp = conf_matrix.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print("\nPER-CLASS BREAKDOWN:")
    print(f"True Negatives: {tn}, False Positives: {fp}, False Negatives: {fn}, True Positives: {tp}")
    print(f"\nSensitivity (Recall for Malignant): {sensitivity:.4f} ({sensitivity*100:.2f}%)")
    print(f"Specificity (Recall for Benign): {specificity:.4f} ({specificity*100:.2f}%)")

    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'{model_name} - Test Set Performance', fontsize=16)

    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                xticklabels=class_names, yticklabels=class_names)
    axes[0, 0].set_title(f'Confusion Matrix (Test Set)\nAccuracy: {accuracy:.2%}')

    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'Sensitivity', 'Specificity']
    metrics_values = [accuracy, precision, recall, f1, sensitivity, specificity]
    bars = axes[0, 1].bar(metrics_names, metrics_values, color=['skyblue', 'lightgreen', 'coral', 'gold', 'lightpink', 'lightcyan'])
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].set_title('Performance Metrics (Test Set)')
    axes[0, 1].axhline(y=0.7, color='gray', linestyle='--')

    for bar, val in zip(bars, metrics_values):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)

    # ROC Curve
    if len(np.unique(all_labels)) == 2 and all_probs.shape[1] >= 2:
        fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
        roc_auc = auc(fpr, tpr)

        axes[1, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
        axes[1, 0].plot([0, 1], [0, 1], 'k--', label='Random')
        axes[1, 0].set_xlabel('False Positive Rate')
        axes[1, 0].set_ylabel('True Positive Rate')
        axes[1, 0].set_title(f'ROC Curve (Test Set)')
        axes[1, 0].legend()
        axes[1, 0].grid(True)

        print(f"\nROC-AUC Score: {roc_auc:.4f}")
    else:
        axes[1, 0].text(0.5, 0.5, 'ROC-AUC not available', ha='center', va='center')
        axes[1, 0].set_title('ROC Curve')
        roc_auc = 0.0

    # Remove empty subplot or add prediction distribution
    axes[1, 1].hist(all_probs[all_labels==0][:, 1] if all_probs.shape[1] >= 2 else all_probs,
                    bins=20, alpha=0.5, label='Benign', color='blue')
    axes[1, 1].hist(all_probs[all_labels==1][:, 1] if all_probs.shape[1] >= 2 else all_probs,
                    bins=20, alpha=0.5, label='Malignant', color='red')
    axes[1, 1].set_xlabel('Probability of Malignant')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Prediction Distribution')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(f'/content/explanations/{model_name}_test_evaluation.png', dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'f1': f1, 'sensitivity': sensitivity, 'specificity': specificity,
        'roc_auc': roc_auc, 'conf_matrix': conf_matrix.tolist()
    }

# Visulize
def visualize_sample_predictions(model, model_name, test_loader, device='cuda'):
    print("\n" + "="*50)
    print(f"SAMPLE PREDICTIONS (TEST SET) - {model_name}")
    print("="*50)

    def denormalize(tensor):
        tensor = tensor.clone().detach().cpu()
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        for t, m, s in zip(tensor, mean, std):
            t.mul_(s).add_(m)
        return tensor

    model.eval()
    benign_samples, malignant_samples = [], []
    benign_labels, malignant_labels = [], []
    benign_preds, malignant_preds = [], []
    benign_probs, malignant_probs = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            for i in range(len(labels)):
                if labels[i] == 0 and len(benign_samples) < 2:
                    benign_samples.append(images[i].cpu())
                    benign_labels.append(labels[i].cpu())
                    benign_preds.append(predicted[i].cpu())
                    benign_probs.append(probs[i].cpu().numpy())
                elif labels[i] == 1 and len(malignant_samples) < 3:
                    malignant_samples.append(images[i].cpu())
                    malignant_labels.append(labels[i].cpu())
                    malignant_preds.append(predicted[i].cpu())
                    malignant_probs.append(probs[i].cpu().numpy())

            if len(benign_samples) == 2 and len(malignant_samples) == 3:
                break

    all_samples = benign_samples + malignant_samples
    all_labels = benign_labels + malignant_labels
    all_preds = benign_preds + malignant_preds
    all_probs = benign_probs + malignant_probs

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f'{model_name} - Sample Predictions (Test Set): Benign (0) vs Malignant (1)', fontsize=16)

    class_names = ['Benign', 'Malignant']

    for i, ax in enumerate(axes):
        img = denormalize(all_samples[i])
        img = np.clip(img.numpy().transpose(1, 2, 0), 0, 1)

        ax.imshow(img)
        ax.axis('off')

        true_label = all_labels[i].item()
        pred_label = all_preds[i].item()
        confidence = all_probs[i][pred_label] * 100

        title_color = 'green' if pred_label == true_label else 'red'
        title = f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.1f}%'
        ax.set_title(title, color=title_color, fontsize=10)

    plt.tight_layout()
    plt.savefig(f'/content/explanations/{model_name}_test_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Sample predictions saved as '/content/explanations/{model_name}_test_predictions.png'")

In [ ]:
# Main function with Stratified K-Fold
def run_all_experiments_stratified(models_config, full_data, n_splits=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    from sklearn.model_selection import StratifiedShuffleSplit

    # Get labels for stratification
    labels = [label for _, label in full_data.samples]

    # Single train/val/test split with stratification (70/15/15)
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))

    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)  # 0.5 of 30% = 15%
    val_idx_rel, test_idx_rel = next(sss2.split(np.zeros(len(temp_idx)), temp_labels))

    val_idx = [temp_idx[i] for i in val_idx_rel]
    test_idx = [temp_idx[i] for i in test_idx_rel]

    # Create datasets
    train_data = Subset(full_data, train_idx)
    val_data = Subset(full_data, val_idx)
    test_data = Subset(full_data, test_idx)

    # Verify stratification
    def print_class_dist(dataset, name):
        labels = [full_data.samples[i][1] for i in dataset.indices]
        benign_pct = 100 * labels.count(0) / len(labels)
        malignant_pct = 100 * labels.count(1) / len(labels)
        print(f"{name}: Benign={benign_pct:.1f}%, Malignant={malignant_pct:.1f}%")

    print(f"\nDataset Split (Stratified):")
    print_class_dist(train_data, "Training")
    print_class_dist(val_data, "Validation")
    print_class_dist(test_data, "Test")

    all_results = {}

    for config in models_config:
        print(f"\n{'='*60}")
        print(f"Processing {config['name']}")
        print(f"{'='*60}")

        # Set transform based on model type for training only
        # if config['transform'] == 'heavy':
        #     train_data.dataset.transform = heavy_transform
        # elif config['transform'] == 'mid':

        # else:
        #     train_data.dataset.transform = simple_transform

        train_data.dataset.transform = mid_transform
        val_data.dataset.transform = val_transform
        test_data.dataset.transform = val_transform

        train_loader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_data, batch_size=8, shuffle=False, num_workers=2)
        test_loader = DataLoader(test_data, batch_size=8, shuffle=False, num_workers=2)

        # Rest of your training/evaluation code remains the same
        checkpoint = checkpointer.load_checkpoint(config['name'], best=True)

        if checkpoint:
          print(f"Found existing checkpoint")
          model = config['class'](**config['params']).to(device)
          model.load_state_dict(checkpoint['model_state_dict'])
          model.eval()

          # Load history
          history_path = f'/content/model_checkpoints/{config["name"]}_history.json'
          if os.path.exists(history_path):
              with open(history_path, 'r') as f:
                  history = json.load(f)
          else:
              history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
        else:
          print(f"Training new model...")
          model = config['class'](**config['params']).to(device)
          history = train_model(model, config['name'], train_loader, val_loader, epochs=15, device=device)
          model.eval()

        # Store and evaluate as before
        all_results[config['name']] = {'model': model, 'history': history, 'test_loader': test_loader}

        metrics = evaluate_model(model, config['name'], test_loader, device)
        visualize_sample_predictions(model, config['name'], test_loader, device)

    return all_results

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
from torchvision.datasets import ImageFolder

# Load all images first
full_data = ImageFolder(root='/content/MINI-DDSM')

# Filter out mask files
filtered_samples = [(path, label) for path, label in full_data.samples if '_Mask' not in path]

# Replace samples
full_data.samples = filtered_samples
full_data.targets = [label for _, label in filtered_samples]

print(f"Total images (excluding masks): {len(full_data)}")

Total images (excluding masks): 5490


In [ ]:
# Run Everything
if __name__ == "__main__":
    # Create directories
    os.makedirs('/content/model_checkpoints', exist_ok=True)
    os.makedirs('/content/explanations', exist_ok=True)

    # Activate keep-alive
    keep_colab_active()

    # Run experiments
    results = run_all_experiments_stratified(models_config, full_data, n_splits=5)

In [ ]:
# Saving all checkpoints to Google Drive
drive_path = '/content/drive/MyDrive/model_checkpoints/'
os.makedirs(drive_path, exist_ok=True)

model_names = ['PureMamba', 'HybridMamba', 'HybridMambaRegularized', 'DenseNetMamba']

# model_names = ['DenseNetMamba']

for model_name in model_names:
    source = f'/content/model_checkpoints/{model_name}_best.pth'
    dest = f'{drive_path}/{model_name}_best.pth'

    if os.path.exists(source):
        shutil.copy(source, dest)
        print(f"Saved {model_name} to Google Drive")
    else:
        print(f"{model_name} checkpoint not found")

In [ ]:
# Loading Checkpoints
os.makedirs('/content/model_checkpoints', exist_ok=True)

# Path to your checkpoints in Drive
drive_checkpoint_path = '/content/drive/MyDrive/model_checkpoints/'

# List all models you want to use
model_names = ['HybridMamba', 'HybridMambaRegularized', 'DenseNetMamba', 'PureMamba']

# model_names = ['DenseNetMamba']

# Copy checkpoints from Drive to local
for model_name in model_names:
    source = f'{drive_checkpoint_path}/{model_name}_best.pth'
    dest = f'/content/model_checkpoints/{model_name}_best.pth'

    if os.path.exists(source):
        shutil.copy(source, dest)
        print(f"✅ Copied {model_name} checkpoint")

        # Check file size to verify
        size = os.path.getsize(dest) / (1024 * 1024)
        print(f"   Size: {size:.2f} MB")
    else:
        print(f"{model_name} checkpoint not found at {source}")

print("\nAll checkpoints imported!")

Generalization

In [ ]:
# BREAST MNIST DATASET LOADER
class BreastMNISTDataset(Dataset):

    def __init__(self, data_path='/content/drive/MyDrive/breastmnist_64.npz', transform=None):
        self.transform = transform

        # Load dataset
        data = np.load(data_path)

        print(f"Loading Breast MNIST from: {data_path}")
        print(f"Available keys: {list(data.keys())}")

        # Try different possible key names
        if 'image' in data.keys():
            self.images = data['image']
        elif 'images' in data.keys():
            self.images = data['images']
        elif 'x' in data.keys():
            self.images = data['x']
        elif 'data' in data.keys():
            self.images = data['data']
        else:
            # If no standard key, use the first array
            first_key = list(data.keys())[0]
            self.images = data[first_key]
            print(f"Using '{first_key}' as image data")

        if 'label' in data.keys():
            self.labels = data['label']
        elif 'labels' in data.keys():
            self.labels = data['labels']
        elif 'y' in data.keys():
            self.labels = data['y']
        else:
            # Try to find labels in second key
            if len(data.keys()) > 1:
                second_key = list(data.keys())[1]
                self.labels = data[second_key]
                print(f"Using '{second_key}' as label data")
            else:
                raise KeyError("No label data found in NPZ file")

        # Handle shape
        print(f"Original images shape: {self.images.shape}")
        print(f"Original labels shape: {self.labels.shape}")

        # Reshape if needed
        if len(self.images.shape) == 3:
            # Assuming (n_samples, height, width)
            self.images = self.images.reshape(-1, 1, self.images.shape[1], self.images.shape[2])
        elif len(self.images.shape) == 4:
            # Already has channel dimension
            if self.images.shape[1] > 1 and self.images.shape[1] <= 4:
                pass  # Keep as is
            else:
                # Maybe channels last
                self.images = self.images.transpose(0, 3, 1, 2)

        # Normalize to [0, 1] if not already
        if self.images.max() > 1.0:
            self.images = self.images.astype(np.float32) / 255.0

        # Ensure labels are 1D
        self.labels = self.labels.flatten()

        # Convert to binary if needed
        unique_labels = np.unique(self.labels)
        if len(unique_labels) > 2:
            print(f"Warning: Found {len(unique_labels)} unique labels. Using first two as benign/malignant")
            # Map first two unique values to 0 and 1
            label_map = {unique_labels[0]: 0, unique_labels[1]: 1}
            self.labels = np.array([label_map[l] for l in self.labels])

        print(f"\nLoaded Breast MNIST: {len(self.labels)} samples")
        print(f"   Images shape: {self.images.shape}")
        print(f"   Benign (0): {sum(self.labels == 0)}")
        print(f"   Malignant (1): {sum(self.labels == 1)}")

        # Show a few samples
        print(f"\n   Sample labels: {self.labels[:10]}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]

        # Ensure 3 channels (RGB) for pretrained models
        if img.shape[0] == 1:
            img = np.repeat(img, 3, axis=0)
        elif img.shape[0] == 3:
            pass  # Already RGB
        elif img.shape[0] > 3:
            img = img[:3]  # Take first 3 channels

        img = torch.FloatTensor(img)

        # Resizing to 256x256
        if self.transform:
            img = self.transform(img)
        else:
            # Default resize
            img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear').squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label, f"sample_{idx}", "BREAST"

    def show_sample(self, idx):

        img = self.images[idx]
        if img.shape[0] > 1:
            img = img[0]  # Take first channel
        plt.figure(figsize=(4, 4))
        plt.imshow(img, cmap='gray')
        plt.title(f"Label: {'Malignant' if self.labels[idx] == 1 else 'Benign'}")
        plt.axis('off')
        plt.show()

In [ ]:
# TEST MODELS ON BREAST MNIST

def test_models_on_breast_mnist(models_config, checkpointer, device):

    print("\n" + "="*70)
    print("TESTING MODELS ON BREAST MNIST")
    print("="*70)

    # Simple transform for Breast MNIST
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Create dataset
    try:
        dataset = BreastMNISTDataset(transform=transform)
    except Exception as e:
        print(f"Failed to load dataset: {e}")
        return {}

    # Load Data
    test_loader = DataLoader(dataset, batch_size=8, shuffle=False)

    print(f"\nTest set: {len(dataset)} samples")

    # Test each model
    results = {}

    for config in models_config:
        print(f"\nTesting {config['name']} on Breast MNIST...")

        # Load checkpoint
        checkpoint = checkpointer.load_checkpoint(config['name'], best=True)
        if not checkpoint:
            checkpoint = checkpointer.load_checkpoint(config['name'].replace('_', ''), best=True)

        if not checkpoint:
            print(f"No checkpoint found for {config['name']}")
            continue

        # Create model
        model = config['class'](**config['params']).to(device)

        # Load weights
        try:
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            print(f"Model loaded")
        except Exception as e:
            print(f"Error loading weights: {e}")
            continue

        model.eval()

        # Test
        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for images, labels, _, _ in tqdm(test_loader, desc=f"Testing {config['name']}"):
                images = images.to(device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                _, predicted = outputs.max(1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
                all_probs.extend(probs.cpu().numpy())

        # Calculate metrics
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
        conf_matrix = confusion_matrix(all_labels, all_preds)

        try:
            probs_positive = [p[1] for p in all_probs]
            auc = roc_auc_score(all_labels, probs_positive)
        except:
            auc = 0.0

        results[config['name']] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc,
            'conf_matrix': conf_matrix
        }

        print(f"\n    Results on Breast MNIST:")
        print(f"      Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"      Precision: {precision:.4f}")
        print(f"      Recall:    {recall:.4f}")
        print(f"      F1-Score:  {f1:.4f}")
        print(f"      AUC:       {auc:.4f}")
        print(f"\n   Confusion Matrix:")
        print(f"      [[{conf_matrix[0,0]:3d} {conf_matrix[0,1]:3d}]")
        print(f"       [{conf_matrix[1,0]:3d} {conf_matrix[1,1]:3d}]]")

    return results

In [ ]:
# RUN BREAST MNIST TEST

print("\n" + "="*70)
print("MAMBASEGX GENERALIZATION TEST")
print("="*70)

try:
    breast_mnist_results = test_models_on_breast_mnist(models_config, checkpointer, device)

    print("\n" + "="*70)
    print("Breast MNIST Results")
    print("="*70)

    for model_name in breast_mnist_results.keys():
        if model_name in results:
            print(f"\n{model_name}:")
            print(f"   Breast MNIST:  Acc={breast_mnist_results[model_name]['accuracy']:.3f}, F1={breast_mnist_results[model_name]['f1']:.3f}")
except NameError as e:
    print(f"Error: {e}")

Segmentation

In [ ]:
# Segmentation Head Training for MINI-DDSM using segmentation_models_pytorch unet architecture, with view specific augmentation

class ViewSpecificAugmentation:

    def __init__(self, view):
        self.view = view

        if view == 'MLO':
            # MLO more rotation
            self.transform = transforms.Compose([
                transforms.RandomRotation(15),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
                transforms.RandomHorizontalFlip(p=0.5),
            ])
        else:
            # CC more flipping
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.3),
                transforms.RandomRotation(5),
            ])

    def __call__(self, img):
        return self.transform(img)

#Data Processing cropping Images for better focus on lesion
class MiniDDSMSegmentationDataset(Dataset):

    def __init__(self, root_dir, transform=None, mask_transform=None, crop_size=256,
                 focus_on_lesion=True, augment=False):
        self.root_dir = root_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.crop_size = crop_size
        self.focus_on_lesion = focus_on_lesion
        self.augment = augment
        self.samples = []

        # Find all mask files
        mask_pattern = os.path.join(root_dir, '**', '*_Mask.jpg')
        mask_files = glob.glob(mask_pattern, recursive=True)

        print(f"Found {len(mask_files)} mask files")

        for mask_path in mask_files:
            img_path = mask_path.replace('_Mask.jpg', '.jpg')

            if os.path.exists(img_path):
                # Load mask to check if it's non-empty
                mask_pil = Image.open(mask_path).convert('L')
                mask_np = np.array(mask_pil)

                # Skip empty masks (no lesion)
                if mask_np.max() == 0:
                    continue

                # Extract info from filename
                filename = os.path.basename(img_path)
                view = 'CC' if 'CC' in filename else 'MLO' if 'MLO' in filename else 'UNKNOWN'
                side = 'LEFT' if 'LEFT' in filename else 'RIGHT' if 'RIGHT' in filename else 'UNKNOWN'

                # Extract label from parent folder
                parent_folder = os.path.basename(os.path.dirname(img_path))
                label = 1 if 'Malignant' in parent_folder else 0

                # Get bounding box of mask
                y_indices, x_indices = np.where(mask_np > 0)
                if len(y_indices) > 0:
                    y_min, y_max = y_indices.min(), y_indices.max()
                    x_min, x_max = x_indices.min(), x_indices.max()

                    # Add padding
                    h, w = mask_np.shape
                    padding = 20
                    y_min = max(0, y_min - padding)
                    y_max = min(h, y_max + padding)
                    x_min = max(0, x_min - padding)
                    x_max = min(w, x_max + padding)
                else:
                    y_min, y_max, x_min, x_max = 0, 256, 0, 256

                self.samples.append({
                    'image': img_path,
                    'mask': mask_path,
                    'label': label,
                    'patient': filename,
                    'view': view,
                    'side': side,
                    'bbox': (x_min, y_min, x_max, y_max)
                })

        print(f"Loaded {len(self.samples)} non-empty image-mask pairs")

        # Count views
        mlo_count = sum(1 for s in self.samples if s['view'] == 'MLO')
        cc_count = sum(1 for s in self.samples if s['view'] == 'CC')
        print(f"Views: MLO={mlo_count}, CC={cc_count}")

        # Count lesion sizes
        if len(self.samples) > 0:
            areas = []
            for s in self.samples[:min(100, len(self.samples))]:
                mask = Image.open(s['mask']).convert('L')
                mask_np = np.array(mask)
                areas.append((mask_np > 0).sum())
            print(f"Average mask area: {np.mean(areas):.1f} pixels")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image and mask
        image = Image.open(sample['image']).convert('RGB')
        mask = Image.open(sample['mask']).convert('L')

        if self.focus_on_lesion:
            # Crop around the lesion
            x_min, y_min, x_max, y_max = sample['bbox']
            image = image.crop((x_min, y_min, x_max, y_max))
            mask = mask.crop((x_min, y_min, x_max, y_max))

        # Apply view-specific augmentation
        if self.augment:
            view_aug = ViewSpecificAugmentation(sample['view'])
            image = view_aug(image)
            if sample['view'] == 'CC':

                if np.random.random() > 0.5:
                    image = transforms.functional.hflip(image)
                    mask = transforms.functional.hflip(mask)

        # Resize to fixed size
        image = image.resize((self.crop_size, self.crop_size), Image.BILINEAR)
        mask = mask.resize((self.crop_size, self.crop_size), Image.NEAREST)

        # Convert to tensor
        if self.transform:
            image = self.transform(image)
        else:
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            image = transform(image)

        if self.mask_transform:
            mask = self.mask_transform(mask)
        else:
            transform = transforms.Compose([
                transforms.ToTensor()
            ])
            mask = transform(mask)
            mask = (mask > 0).float()  # Binarize

        label = torch.tensor(sample['label'], dtype=torch.long)

        return image, mask, label, sample['patient'], sample['view'], sample['side']

In [ ]:
class ModelWithSegmentation(nn.Module):
    def __init__(self, base_model, model_name, num_classes=1):
        super().__init__()
        self.base_model = base_model
        self.model_name = model_name

        # SMP segmentation head
        self.seg_head = smp.Unet(
            encoder_name='resnet18',
            encoder_weights=None,
            in_channels=384,
            classes=num_classes,
            activation=None
        )

        # Freeze base model
        for param in self.base_model.parameters():
            param.requires_grad = False
        print("Base model frozen - training only segmentation head")

    def get_features(self, x):
        if 'PureMamba' in self.model_name:
            return self.base_model.patch_embed(x)
        elif 'DenseNetMamba' in self.model_name:
            features = self.base_model.classification_model.encoder(x)[-1]
            return self.base_model.classification_model.projection(features)
        else:
            features = self.base_model.classification_model.encoder(x)
            return self.base_model.classification_model.projection(features)

    def forward(self, x):
        features = self.get_features(x)
        # For classification - handle both MLP and conv heads
        # Use the base model's forward for classification
        cls_output = self.base_model(x)

        seg_output = self.seg_head(features)
        return cls_output, seg_output

In [ ]:
# Training
@timer
def train_segmentation_head(model, model_name, train_loader, val_loader, epochs=15, device='cuda'):

    criterion = smp.losses.DiceLoss(mode='binary', from_logits=True)
    optimizer = optim.Adam(model.seg_head.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_dice = 0
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}

    print(f"\n{'='*50}")
    print(f"Training Segmentation Head for {model_name}")
    print(f"{'='*50}")

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, train_dice, train_batches = 0, 0, 0

        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for images, masks, _, _, _, _ in train_bar:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            _, seg_output = model(images)

            if seg_output.shape[-2:] != masks.shape[-2:]:
                seg_output = F.interpolate(seg_output, size=masks.shape[-2:],
                                          mode='bilinear', align_corners=False)

            loss = criterion(seg_output, masks)
            loss.backward()
            optimizer.step()

            seg_probs = torch.sigmoid(seg_output)
            seg_pred = (seg_probs > 0.3).float()

            intersection = (seg_pred * masks).sum()
            pred_sum = seg_pred.sum()
            mask_sum = masks.sum()

            if mask_sum == 0:
                dice = 1.0 if pred_sum == 0 else 0.0
            else:
                dice = (2.0 * intersection + 1e-8) / (pred_sum + mask_sum + 1e-8)
                dice = dice.item()

            train_loss += loss.item()
            train_dice += dice
            train_batches += 1
            train_bar.set_postfix({'loss': f'{loss.item():.4f}', 'dice': f'{dice:.4f}'})

        # Validation
        model.eval()
        val_loss, val_dice, val_batches = 0, 0, 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            for images, masks, _, _, _, _ in val_bar:
                images, masks = images.to(device), masks.to(device)
                _, seg_output = model(images)

                if seg_output.shape[-2:] != masks.shape[-2:]:
                    seg_output = F.interpolate(seg_output, size=masks.shape[-2:],
                                              mode='bilinear', align_corners=False)

                loss = criterion(seg_output, masks)

                seg_probs = torch.sigmoid(seg_output)
                seg_pred = (seg_probs > 0.3).float()

                intersection = (seg_pred * masks).sum()
                pred_sum = seg_pred.sum()
                mask_sum = masks.sum()

                if mask_sum == 0:
                    dice = 1.0 if pred_sum == 0 else 0.0
                else:
                    dice = (2.0 * intersection + 1e-8) / (pred_sum + mask_sum + 1e-8)
                    dice = dice.item()

                val_loss += loss.item()
                val_dice += dice
                val_batches += 1

        train_loss /= train_batches
        train_dice /= train_batches
        val_loss /= val_batches
        val_dice /= val_batches

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)

        print(f'\nEpoch {epoch+1} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        print(f'          Train Dice: {train_dice:.4f}, Val Dice: {val_dice:.4f}')

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.seg_head.state_dict(), f'/content/{model_name}_seg_head_best.pth')
            print(f'Saved best segmentation head (val_dice: {val_dice:.4f})')

    return history

# Training Loop
def train_all_segmentation_heads(models_config, checkpointer, train_loader, val_loader):

    results = {}
    all_histories = {}

    for config in models_config:
        print(f"\n{'='*60}")
        print(f"Processing {config['name']}")
        print(f"{'='*60}")

        # Load checkpoint
        checkpoint = checkpointer.load_checkpoint(config['name'], best=True)
        if not checkpoint:
            checkpoint = checkpointer.load_checkpoint(config['name'].replace('_', ''), best=True)

        if not checkpoint:
            print(f"No checkpoint found for {config['name']}")
            continue

        model = config['class'](**config['params']).to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("Loaded trained classification model")

        model_with_seg = ModelWithSegmentation(model, config['name']).to(device)

        history,_ = train_segmentation_head(
            model_with_seg, config['name'], train_loader, val_loader,
            epochs=15, device=device
        )

        best_dice = max(history['val_dice'])

        torch.save({
            'base_model_state_dict': model.state_dict(),
            'seg_head_state_dict': model_with_seg.seg_head.state_dict(),
            'history': history
        }, f'/content/{config["name"]}_with_segmentation.pth')

        results[config['name']] = history
        all_histories[config['name']] = history
        print(f"Saved {config['name']} with segmentation head")

    for name, history in results.items():
        best_dice = max(history['val_dice'])
        print(f"{name}: Best Val Dice = {best_dice:.4f}")

    return results

In [ ]:
# Dataset Setup
base_dir = '/content/MINI-DDSM'

# Create transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.ToTensor()
])

# Load dataset
full_dataset = MiniDDSMSegmentationDataset(
    base_dir,
    transform=transform,
    mask_transform=mask_transform,
    crop_size=256,
    focus_on_lesion=True,
    augment=True  # View-specific augmentation works per sample
)

# Split by patient
from sklearn.model_selection import train_test_split

# Extract unique patient IDs
unique_patients = list(set([s['patient'].split('_')[0] for s in full_dataset.samples]))

# Get labels for stratification
patient_labels = []
for patient in unique_patients:
    for s in full_dataset.samples:
        if s['patient'].startswith(patient):
            patient_labels.append(s['label'])
            break

# Split patients into train/val/test (70/15/15)
train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.3,
    random_state=42,
    stratify=patient_labels
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    random_state=42,
    stratify=[patient_labels[unique_patients.index(p)] for p in temp_patients]
)

# Create indices based on patient split
train_indices = []
val_indices = []
test_indices = []

for i, sample in enumerate(full_dataset.samples):
    patient_id = sample['patient'].split('_')[0]
    if patient_id in train_patients:
        train_indices.append(i)
    elif patient_id in val_patients:
        val_indices.append(i)
    else:
        test_indices.append(i)

print(f"\nSplit results (by patient):")
print(f"   Train: {len(train_indices)} samples from {len(train_patients)} patients")
print(f"   Val:   {len(val_indices)} samples from {len(val_patients)} patients")
print(f"   Test:  {len(test_indices)} samples from {len(test_patients)} patients")

# Count views per split
train_mlo = sum(1 for i in train_indices if full_dataset.samples[i]['view'] == 'MLO')
train_cc = sum(1 for i in train_indices if full_dataset.samples[i]['view'] == 'CC')
val_mlo = sum(1 for i in val_indices if full_dataset.samples[i]['view'] == 'MLO')
val_cc = sum(1 for i in val_indices if full_dataset.samples[i]['view'] == 'CC')
test_mlo = sum(1 for i in test_indices if full_dataset.samples[i]['view'] == 'MLO')
test_cc = sum(1 for i in test_indices if full_dataset.samples[i]['view'] == 'CC')

print(f"\nView distribution:")
print(f"   Train - MLO: {train_mlo}, CC: {train_cc}")
print(f"   Val   - MLO: {val_mlo}, CC: {val_cc}")
print(f"   Test  - MLO: {test_mlo}, CC: {test_cc}")

# Create datasets
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
# Run Everything

if __name__ == "__main__":
    print("Starting Segmentation Head Training on MINI-DDSM")
    print("="*60)

    results = train_all_segmentation_heads(models_config, checkpointer, train_loader, val_loader)

In [ ]:
#Load Models
def load_all_saved_models():

    torch.manual_seed(42)
    np.random.seed(42)

    models = {}

    for config in models_config:
        model_name = config['name']
        checkpoint_path = f'/content/{model_name}_with_segmentation.pth'

        if os.path.exists(checkpoint_path):
            print(f"Loading {model_name}...")

            # Load checkpoint
            checkpoint = torch.load(checkpoint_path)

            # Load base model
            base_model = config['class'](**config['params']).to(device)
            base_model.load_state_dict(checkpoint['base_model_state_dict'])

            # Add segmentation head
            model_with_seg = ModelWithSegmentation(base_model, model_name).to(device)
            model_with_seg.seg_head.load_state_dict(checkpoint['seg_head_state_dict'])
            model_with_seg.eval()

            # Get training history
            history = checkpoint['history']
            best_dice = max(history['val_dice'])

            models[model_name] = {
                'model': model_with_seg,
                'history': history,
                'best_dice': best_dice
            }

            print(f"   Best Val Dice: {best_dice:.4f}")
        else:
            print(f"{model_name} not found at {checkpoint_path}")

    return models

In [ ]:
# Visulize Segementation Training


def compare_model_predictions(models, test_loader, num_samples=3):

    # Set models to eval mode
    for model_data in models.values():
        model_data['model'].eval()

    # Explicitly sample both MLO and CC
    fixed_samples = []
    mlo_samples = []
    cc_samples = []

    with torch.no_grad():
        for images, masks, labels, patients, views, sides in test_loader:
            for i in range(len(images)):
                view = views[i]
                sample = {
                    'image': images[i:i+1].to(device),
                    'mask': masks[i:i+1],
                    'patient': patients[i],
                    'view': view,
                    'side': sides[i],
                    'label': labels[i].item()
                }

                if view == 'MLO' and len(mlo_samples) < num_samples:
                    mlo_samples.append(sample)
                elif view == 'CC' and len(cc_samples) < num_samples:
                    cc_samples.append(sample)

                if len(mlo_samples) >= num_samples and len(cc_samples) >= num_samples:
                    break

            if len(mlo_samples) >= num_samples and len(cc_samples) >= num_samples:
                break

    fixed_samples = []
    for i in range(num_samples):
        if i < len(mlo_samples):
            fixed_samples.append(mlo_samples[i])
        if i < len(cc_samples):
            fixed_samples.append(cc_samples[i])

    # Visualize
    fig, axes = plt.subplots(len(fixed_samples), len(models) + 1,
                             figsize=(5*(len(models)+1), 5*len(fixed_samples)))

    if len(fixed_samples) == 1:
        axes = axes.reshape(1, -1)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

    for i, sample in enumerate(fixed_samples):
        img = sample['image']
        mask = sample['mask']
        label = sample['label']
        lesion_type = 'Malignant' if label == 1 else 'Benign'

        img_np = torch.clamp(img.cpu() * std.cpu() + mean.cpu(), 0, 1).squeeze().permute(1, 2, 0).numpy()
        mask_np = mask.cpu().squeeze().numpy()

        # Ground truth
        axes[i, 0].imshow(img_np)
        axes[i, 0].imshow(mask_np, cmap='jet', alpha=0.6)
        view_color = 'red' if sample['view'] == 'MLO' else 'blue'

        title = f'Ground Truth\n{sample["patient"]}\n{sample["view"]} - {sample["side"]}\n{lesion_type}'
        axes[i, 0].set_title(title, color=view_color)
        axes[i, 0].axis('off')

        # Predictions for each model
        for j, (model_name, model_data) in enumerate(models.items()):
            model = model_data['model']
            with torch.no_grad():
                _, seg_output = model(sample['image'])

            seg_output = F.interpolate(seg_output, size=(256, 256),
                                      mode='bilinear', align_corners=False)
            seg_probs = torch.sigmoid(seg_output)
            seg_pred = (seg_probs > 0.01).float()

            pred_np = seg_pred[0].cpu().squeeze().numpy()
            pred_np = binary_dilation(pred_np, iterations=3).astype(float)

            axes[i, j+1].imshow(img_np)
            if pred_np.sum() > 0:
                axes[i, j+1].imshow(pred_np, cmap='jet', alpha=0.6)
                confidence = seg_probs.max().item()
                axes[i, j+1].text(10, 30, f'Conf: {confidence:.2f}',
                                 color='white', fontsize=9, weight='bold',
                                 bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7))

            pred_title = f'{model_name}\nDice: {model_data["best_dice"]:.3f}\nTrue: {lesion_type}'
            axes[i, j+1].set_title(pred_title)
            axes[i, j+1].axis('off')

    plt.tight_layout()
    plt.savefig('/content/all_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved comparison to /content/all_models_comparison.png")
    print(f"{[s['view'] for s in fixed_samples]}")

In [ ]:
# Run Everything

if __name__ == "__main__":
    print("Loading all saved segmentation models...")
    print("="*60)

    # Load all models
    models = load_all_saved_models()

    if not models:
        print("No saved models found! Run training first.")
    else:
        print(f"\nLoaded {len(models)} models")

        # Compare all models side by side
        compare_model_predictions(models, test_loader, num_samples=3)

        # Print summary
        print("\n" + "="*60)
        print("MODEL COMPARISON SUMMARY")
        print("="*60)
        for name, data in sorted(models.items(), key=lambda x: x[1]['best_dice'], reverse=True):
            print(f"{name}: {data['best_dice']:.4f}")

Explainability

In [ ]:
class SimpleGradCAM:
    def __init__(self, model, target_layer_name=None, alpha=0.5):
        self.model = model
        self.alpha = alpha
        self.target_layer = None
        self.gradients = None
        self.activations = None

        # Find appropriate target layer
        if target_layer_name:
            self.target_layer = self._find_layer_by_name(target_layer_name)
        else:
            self.target_layer = self._get_best_target_layer()

        if self.target_layer is None:
            raise ValueError("Could not find suitable target layer for Grad-CAM")

        # Register hooks
        self.handle_forward = self.target_layer.register_forward_hook(self._save_activation)
        self.handle_backward = self.target_layer.register_full_backward_hook(self._save_gradient)

        print(f"Using target layer: {self.target_layer.__class__.__name__}")
        if hasattr(self.target_layer, 'in_channels'):
            print(f"  Input channels: {self.target_layer.in_channels}")
        if hasattr(self.target_layer, 'out_channels'):
            print(f"  Output channels: {self.target_layer.out_channels}")

    def _find_layer_by_name(self, layer_name):
        """Find a specific layer by name"""
        for name, module in self.model.named_modules():
            if name == layer_name or module.__class__.__name__ == layer_name:
                print(f"Found target layer: {name}")
                return module
        return None

    def _get_best_target_layer(self):
      # Priority: projection > patch_embed > last Conv2d > last Conv1d
      priority = ['projection', 'patch_embed']

      named = dict(self.model.named_modules())

      for name in priority:
          for key, module in named.items():
              if key.endswith(name) and isinstance(module, nn.Conv2d):
                  print(f"Using: {key}")
                  return module

      # Fallback: walk all modules
      last_conv = None
      for module in self.model.modules():
          if isinstance(module, (nn.Conv2d, nn.Conv1d)):
              last_conv = module

      if last_conv:
          print(f"Fallback: {last_conv.__class__.__name__} ({last_conv.in_channels}→{last_conv.out_channels})")

      return last_conv


    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        """Generate Grad-CAM heatmap"""
        self.model.eval()
        input_tensor.requires_grad_(True)

        # Forward pass
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        if self.gradients is None or self.activations is None:
            print("No gradients or activations captured")
            return None, class_idx

        # Handle different tensor shapes
        heatmap = self._compute_heatmap()

        if heatmap is None:
            return None, class_idx

        # Post-process heatmap
        heatmap = F.relu(heatmap)
        heatmap = heatmap.cpu().detach().numpy()

        if heatmap.ndim == 3:
            heatmap = heatmap[0]

        # Normalize
        if heatmap.max() - heatmap.min() > 1e-8:
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

        return heatmap, class_idx

    def _compute_heatmap(self):
        """Compute heatmap based on activation type"""
        activations = self.activations
        gradients = self.gradients

        # Handle Conv2d [B, C, H, W]
        if len(activations.shape) == 4:
            pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
            weighted_activations = activations * pooled_gradients[None, :, None, None]
            heatmap = torch.mean(weighted_activations, dim=1)
            return heatmap

        # Handle Conv1d [B, C, Seq]
        elif len(activations.shape) == 3:
            pooled_gradients = torch.mean(gradients, dim=[0, 2])
            weighted_activations = activations * pooled_gradients[None, :, None]
            heatmap_1d = torch.mean(weighted_activations, dim=1)

            # Try to reshape to 2D square if possible
            seq_len = heatmap_1d.shape[1]
            h = w = int(seq_len ** 0.5)
            if h * w == seq_len:
                heatmap = heatmap_1d.reshape(1, h, w)
                return heatmap
            else:
                # For non-square sequences, create a square approximation
                target_size = int(np.ceil(np.sqrt(seq_len)))
                padded = F.pad(heatmap_1d, (0, target_size**2 - seq_len))
                heatmap = padded.reshape(1, target_size, target_size)
                # Resize to reasonable dimensions
                heatmap = F.interpolate(heatmap.unsqueeze(0), size=(7, 7), mode='bilinear', align_corners=False).squeeze(0)
                return heatmap

        # Handle Linear layers (less common)
        elif len(activations.shape) == 2:
            print("Warning: Linear layer detected - Grad-CAM works best with conv layers")
            # Convert to pseudo-spatial by taking square root
            seq_len = activations.shape[1]
            h = w = int(np.sqrt(seq_len))
            if h * w == seq_len:
                reshaped_acts = activations.reshape(activations.shape[0], h, w)
                reshaped_grads = gradients.reshape(gradients.shape[0], h, w)
                heatmap = torch.mean(reshaped_grads * reshaped_acts, dim=0)
                return heatmap.unsqueeze(0)

        else:
            print(f"Unexpected activation shape: {activations.shape}")
            return None

    def overlay_heatmap(self, image, heatmap, resize_to_original=True):
        """Create overlay of heatmap on original image"""
        # Ensure image is uint8
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8)

        # Ensure heatmap is 2D
        if heatmap.ndim > 2:
            heatmap = heatmap.squeeze()

        # Resize heatmap to match image if needed
        if resize_to_original:
            heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))

        # Normalize heatmap to [0, 1]
        heatmap = np.clip(heatmap, 0, 1)

        # Convert to uint8 for colormap
        heatmap_uint8 = (heatmap * 255).astype(np.uint8)

        # Apply colormap (creates BGR image)
        heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

        # Convert BGR to RGB
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        # Blend images
        overlay = (self.alpha * heatmap_colored + (1 - self.alpha) * image).astype(np.uint8)

        return overlay

    def explain_image(self, image_path, device='cuda', transform=None, resize_heatmap=True):
        """Generate explanation for a single image"""
        # Load and preprocess image
        original_image = Image.open(image_path).convert('RGB')

        if transform is None:
            raise ValueError("Transform must be provided")

        input_tensor = transform(original_image).unsqueeze(0).to(device)

        # Get prediction
        with torch.no_grad():
            output = self.model(input_tensor)
            probs = torch.softmax(output, dim=1)
            predicted_class = output.argmax(dim=1).item()
            confidence = probs[0][predicted_class].item()

        # Generate heatmap
        heatmap, _ = self.generate(input_tensor, predicted_class)

        if heatmap is None:
            return None

        # Prepare original image (resize to standard size for consistency)
        original_np = np.array(original_image.resize((256, 256)))

        # Resize heatmap to match original if requested
        if resize_heatmap:
            heatmap_resized = cv2.resize(heatmap, (original_np.shape[1], original_np.shape[0]))
        else:
            heatmap_resized = heatmap

        # Create overlay
        overlay = self.overlay_heatmap(original_np, heatmap_resized, resize_to_original=False)

        # Determine classes
        true_class = 'Benign' if 'benign' in image_path.lower() else 'Malignant'
        pred_class = 'Malignant' if predicted_class == 1 else 'Benign'
        is_correct = true_class == pred_class

        return {
            'original': original_np,
            'heatmap': heatmap_resized,
            'overlay': overlay,
            'predicted_class': predicted_class,
            'confidence': confidence,
            'predicted_name': pred_class,
            'true_name': true_class,
            'correct': is_correct
        }

    def remove_hooks(self):
        """Remove registered hooks to prevent memory leaks"""
        if hasattr(self, 'handle_forward'):
            self.handle_forward.remove()
        if hasattr(self, 'handle_backward'):
            self.handle_backward.remove()

# Usage example with your DenseNetMamba model:
def visualize_gradcam_explanations(models_config, image_paths, device, val_transform):
    """Visualize Grad-CAM explanations for multiple models"""

    for config in models_config:
        print(f"\n{'='*60}")
        print(f"{config['name']}")
        print(f"{'='*60}")

        checkpoint_path = f'/content/model_checkpoints/{config["name"]}_best.pth'
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint not found: {checkpoint_path}")
            continue

        # Load model
        model = config['class'](**config['params']).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        # Get model accuracy
        metrics = checkpoint.get('metrics', {})
        val_acc = metrics.get('val_acc', 'N/A')
        print(f"Validation Accuracy: {val_acc if val_acc == 'N/A' else f'{val_acc:.2f}%'}")

        # Initialize Grad-CAM with custom target layer selection
        # You can specify which layer to use:
        # grad_cam = ImprovedGradCAM(model, target_layer_name='conv_head.8')  # specific layer
        grad_cam = SimpleGradCAM(model, alpha=0.5)  # auto-select best layer

        # Generate explanations
        results = []
        for img_path in image_paths:
            result = grad_cam.explain_image(img_path, device=device, transform=val_transform)
            if result:
                results.append(result)
                status = "✓" if result['correct'] else "✗"
                print(f"  {status} {result['true_name']} → {result['predicted_name']} ({result['confidence']:.1%})")

        # Remove hooks to avoid memory issues
        grad_cam.remove_hooks()

        # Create composite figure
        if results:
            fig, axes = plt.subplots(len(results), 3, figsize=(12, 4*len(results)))
            if len(results) == 1:
                axes = axes.reshape(1, -1)

            fig.suptitle(f'{config["name"]} (Val Acc: {val_acc if val_acc == "N/A" else f"{val_acc:.1f}%"})', fontsize=14)

            for i, result in enumerate(results):
                # Original image
                axes[i, 0].imshow(result['original'])
                axes[i, 0].set_title(f'True: {result["true_name"]}', fontsize=10)
                axes[i, 0].axis('off')

                # Heatmap
                axes[i, 1].imshow(result['heatmap'], cmap='jet')
                axes[i, 1].set_title('Grad-CAM', fontsize=10)
                axes[i, 1].axis('off')

                # Overlay
                axes[i, 2].imshow(result['overlay'])
                color = 'green' if result['correct'] else 'red'
                title = f"Pred: {result['predicted_name']}\nConf: {result['confidence']:.1%}"
                axes[i, 2].set_title(title, color=color, fontsize=10)
                axes[i, 2].axis('off')

            plt.tight_layout()
            plt.savefig(f'{config["name"]}_gradcam_explanations.png', dpi=300, bbox_inches='tight')
            plt.show()
            print(f"Saved: {config['name']}_gradcam_explanations.png")
        else:
            print("No valid results generated")


In [ ]:
def get_random_balanced_images(num_images=6):
    """Get balanced random images from both classes, excluding mask files"""
    benign_paths = []
    malignant_paths = []

    for root, dirs, files in os.walk('/content/MINI-DDSM'):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')) and '_Mask' not in file:
                if 'benign' in root.lower():
                    benign_paths.append(os.path.join(root, file))
                elif 'malignant' in root.lower() or 'cancer' in root.lower():
                    malignant_paths.append(os.path.join(root, file))

    print(f"Found (excluding masks): {len(benign_paths)} Benign, {len(malignant_paths)} Malignant")

    random.seed(42)
    num_each = num_images // 2
    selected = []

    if len(benign_paths) >= num_each:
        selected.extend(random.sample(benign_paths, num_each))
    else:
        selected.extend(benign_paths)

    if len(malignant_paths) >= num_each:
        selected.extend(random.sample(malignant_paths, num_each))
    else:
        selected.extend(malignant_paths)

    return selected

In [ ]:
# Get balanced random images
image_paths = get_random_balanced_images(6)  # 3 benign, 3 malignant
print(f"\nSelected {len(image_paths)} images:")
for i, path in enumerate(image_paths):
    label = 'Benign' if 'benign' in path.lower() else 'Malignant'
    print(f"  {i+1}. {label}: {os.path.basename(path)}")

# Modified explanation generation loop
for config in models_config:
    print(f"\n{'='*60}")
    print(f"{config['name']}")
    print(f"{'='*60}")

    checkpoint_path = f'/content/model_checkpoints/{config["name"]}_best.pth'
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found")
        continue

    # Load model
    model = config['class'](**config['params']).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Get model accuracy
    metrics = checkpoint.get('metrics', {})
    val_acc = metrics.get('val_acc', 'N/A')
    print(f"Validation Accuracy: {val_acc if val_acc == 'N/A' else f'{val_acc:.2f}%'}")

    # Initialize Grad-CAM for this model
    grad_cam = SimpleGradCAM(model, alpha=0.5)

    # Generate explanations
    results = []
    for img_path in image_paths:
        result = grad_cam.explain_image(img_path, device=device, transform=val_transform)
        if result:
            results.append(result)
            status = "✓" if result['correct'] else "✗"
            print(f"  {status} {result['true_name']} → {result['predicted_name']} ({result['confidence']:.1%})")

    # Remove hooks to avoid memory issues
    grad_cam.remove_hooks()

    # Create composite figure
    if results:
        fig, axes = plt.subplots(len(results), 3, figsize=(12, 4*len(results)))
        if len(results) == 1:
            axes = axes.reshape(1, -1)

        fig.suptitle(f'{config["name"]} (Val Acc: {val_acc if val_acc == "N/A" else f"{val_acc:.1f}%"})', fontsize=14)

        for i, result in enumerate(results):
            axes[i, 0].imshow(result['original'])
            axes[i, 0].set_title(f'True: {result["true_name"]}', fontsize=10)
            axes[i, 0].axis('off')

            axes[i, 1].imshow(result['heatmap'], cmap='jet')
            axes[i, 1].set_title('Grad-CAM', fontsize=10)
            axes[i, 1].axis('off')

            axes[i, 2].imshow(result['overlay'])
            color = 'green' if result['correct'] else 'red'
            title = f"Pred: {result['predicted_name']}\nConf: {result['confidence']:.1%}"
            axes[i, 2].set_title(title, color=color, fontsize=10)
            axes[i, 2].axis('off')

        plt.tight_layout()
        plt.savefig(f'{config["name"]}_explanations.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: {config['name']}_explanations.png")